# Football Player Positioning AI - LSTM Training

Train a per-player LSTM model on Google Colab with free GPU.

**Important**: Each game's player is a DIFFERENT person. `game1_home_11` != `game2_home_11`.

## 0. Configuration

Set which game and player to train below.

In [ ]:
# ============================================
#  CONFIGURE HERE: which game and player
# ============================================
GAME = 'game1'       # game1, game2, game3
PLAYER = 'home_11'   # e.g. home_11, away_25
DATASET_ID = f'{GAME}_{PLAYER}'
# ============================================

print(f'[CONFIG] Game:   {GAME}')
print(f'[CONFIG] Player: {PLAYER}')
print(f'[CONFIG] ID:     {DATASET_ID}')

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

## 2. Clone Repository

In [ ]:
import os
if os.path.exists('football-positioning-ai'):
    print('[DEBUG] Repo exists, pulling latest...')
    !cd football-positioning-ai && git pull
else:
    !git clone https://github.com/Sysus611/football-positioning-ai.git
%cd football-positioning-ai
!pip install pandas numpy pyarrow scipy pyyaml matplotlib -q
print('[OK] Ready')

## 3. Download Data (only the selected game)

In [ ]:
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"
FILES = {
    'game1': {
        'tracking_home.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv',
    },
    'game2': {
        'tracking_home.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv',
    },
    'game3': {
        'tracking.txt': 'Sample_Game_3/Sample_Game_3_tracking.txt',
        'metadata.xml': 'Sample_Game_3/Sample_Game_3_metadata.xml',
        'metadata.json': 'Sample_Game_3/Sample_Game_3_events.json',
    },
}

# Only download the selected game
file_map = FILES[GAME]
game_dir = f'data/raw/metrica/{GAME}'
os.makedirs(game_dir, exist_ok=True)

print(f'[DEBUG] Downloading {GAME} data...')
for local_name, remote_path in file_map.items():
    dest = f'{game_dir}/{local_name}'
    if os.path.exists(dest):
        print(f'  [SKIP] {local_name} already exists')
        continue
    print(f'  [DOWNLOAD] {remote_path}...', end='')
    urllib.request.urlretrieve(f'{BASE_URL}/{remote_path}', dest)
    print(f' OK ({os.path.getsize(dest)/1024/1024:.1f} MB)')
print('[OK] Download complete')

## 4. Preprocessing (selected game only)

In [ ]:
import time
t0 = time.time()
!python src/preprocessing/preprocess.py
print(f'\n[OK] Preprocessing done in {time.time()-t0:.0f}s')

## 5. Feature Engineering (selected game + player only)

In [ ]:
import time
t0 = time.time()
!python src/features/build_features.py {GAME} {PLAYER}
print(f'\n[OK] Feature engineering done in {time.time()-t0:.0f}s')

## 6. Train

In [ ]:
import time
print(f'[DEBUG] Training {DATASET_ID}...')
t0 = time.time()
!python src/training/train.py {DATASET_ID}
print(f'\n[OK] Training done in {time.time()-t0:.0f}s ({(time.time()-t0)/60:.1f} min)')

## 7. Results & Visualization

In [ ]:
import os, torch
ckpt = torch.load(f'data/models/{DATASET_ID}.pt', map_location='cpu', weights_only=False)
val_loss = ckpt['best_val_loss']
dist_m = (val_loss ** 0.5) * ((105 + 68) / 2)
print(f'[RESULT] {DATASET_ID}')
print(f'  Val Loss: {val_loss:.6f}')
print(f'  ~Distance Error: {dist_m:.2f}m')
print(f'  Best Epoch: {ckpt["epoch"]}')

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt, sys
sys.path.insert(0, '.')
from src.model.lstm_baseline import PlayerLSTM

ckpt = torch.load(f'data/models/{DATASET_ID}.pt', map_location='cpu', weights_only=False)
model = PlayerLSTM(input_dim=ckpt['input_dim'], pred_frames=ckpt['pred_frames'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

data = np.load(f'data/tensors/{DATASET_ID}.npz')
X_val, Y_val = data['X_val'], data['Y_val']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'{DATASET_ID} - Predicted vs Actual (validation)', fontsize=14)

for ax in axes.flat:
    idx = np.random.randint(len(X_val))
    with torch.no_grad():
        pred = model(torch.from_numpy(X_val[idx:idx+1])).numpy()[0]
    actual = Y_val[idx]
    obs_x, obs_y = X_val[idx, :, -4], X_val[idx, :, -3]

    ax.set_xlim(-0.05, 1.05); ax.set_ylim(1.05, -0.05)
    ax.set_facecolor('#2d5a27')
    ax.add_patch(plt.Rectangle((0,0),1,1, fill=False, edgecolor='white', lw=1))
    ax.axvline(x=0.5, color='white', lw=0.5, alpha=0.5)
    ax.plot(obs_x, obs_y, 'c-', lw=1.5, alpha=0.6, label='Past 3s')
    ax.plot(actual[:,0], actual[:,1], 'w-', lw=2, label='Actual')
    ax.plot(pred[:,0], pred[:,1], 'r--', lw=2, label='Predicted')
    err = np.sqrt(((pred-actual)**2).sum(axis=1)).mean() * ((105+68)/2)
    ax.set_title(f'Sample #{idx} (err: {err:.1f}m)', color='white')
    ax.legend(fontsize=8); ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('prediction_vis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Download Model

In [ ]:
from google.colab import files
files.download(f'data/models/{DATASET_ID}.pt')